# 03 - Generate coastal points from the 144 JPS stations

Input: `data/cleaned/jps_coastal_erosion_cleaned_144.csv`.
Output: `data/processed/terengganu_coastal_erosion_transects_144.csv`.

Retain every original station. Generate points at 20-metre intervals only between
non-Protected station pairs in the same coastal segment. A new segment starts at
a coastal-cell boundary or a gap larger than 3 km. Preserve the existing linear
interpolation method and keep source station IDs for traceability.

These interpolated coastal points are not new independent JPS observations and
are not the shore-normal CoastSat transects used for temporal shoreline prediction.


In [1]:
# Pipeline step 1

from pathlib import Path

# Locate the project from either the repository root or its notebooks folder.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data" / "raw" / "jps_coastal_erosion_full.csv").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Open this notebook from inside FYP_Coastal_Erosion.")
    PROJECT_ROOT = PROJECT_ROOT.parent

import pandas as pd
import numpy as np

df = pd.read_csv(
    PROJECT_ROOT / "data/cleaned/jps_coastal_erosion_cleaned_144.csv"
)

print("Dataset shape:", df.shape)
print("Total JPS stations:", len(df))
print("Protected stations:", df["Is_Protected"].sum())

df.head()
assert len(df) == 144 and df["No"].is_unique


Dataset shape: (144, 18)
Total JPS stations: 144
Protected stations: 9


In [2]:
# Pipeline step 2

from pyproj import Geod

# Sort stations by JPS number
df_sorted = df.sort_values("No").reset_index(drop=True)

# Extract coordinates
lons = df_sorted["Longitude_DD"].to_numpy()
lats = df_sorted["Latitude_DD"].to_numpy()

# Calculate distances between consecutive stations
geod = Geod(ellps="WGS84")

_, _, distances = geod.inv(
    lons[:-1], lats[:-1],
    lons[1:], lats[1:]
)

# Convert metres to kilometres
df_sorted["Gap_From_Previous_km"] = np.r_[
    np.nan, distances / 1000
]

print("Total JPS stations:", len(df_sorted))

print("Missing coordinates:")
print(df_sorted[
    ["Latitude_DD", "Longitude_DD"]
].isnull().sum())

print("\nTotal straight-line distance (km):",
      round(distances.sum() / 1000, 2))

print("\n8 largest gaps between stations:")
print(
    df_sorted.nlargest(
        8, "Gap_From_Previous_km"
    )[
        ["No", "Coastal_Cell",
         "Area_Location", "Gap_From_Previous_km"]
    ].round(2)
)


Total JPS stations: 144
Missing coordinates:
Latitude_DD     0
Longitude_DD    0
dtype: int64

Total straight-line distance (km): 225.27

8 largest gaps between stations:
      No Coastal_Cell                      Area_Location  Gap_From_Previous_km
21    22           C3  West of Sungai Merang river mouth                 25.29
107  108           C8               Kampung Teluk Bedara                 11.42
140  141          C11                       Kampung Ibai                  9.49
28    29           C4        North of Kampung Batu Rakit                  9.17
37    38           C5             Kampung Seberang Takir                  8.33
106  107           C8     Sungai Kuala Abang river mouth                  7.33
9     10           C2            North of Kampung Beting                  7.02
133  134          C11   Kelab Golf and Rekreasi Petronas                  6.80


In [3]:
# Pipeline step 3

# Identify large gaps between JPS stations

df_sorted["Previous_No"] = (
    df_sorted["No"].shift(1).astype("Int64")
)

print("Gaps larger than 3 km:",
      (df_sorted["Gap_From_Previous_km"] > 3).sum())

print("Gaps larger than 5 km:",
      (df_sorted["Gap_From_Previous_km"] > 5).sum())

print("Gaps larger than 10 km:",
      (df_sorted["Gap_From_Previous_km"] > 10).sum())

print("\nStations with gaps larger than 3 km:")

print(
    df_sorted.loc[
        df_sorted["Gap_From_Previous_km"] > 3,
        ["Previous_No", "No",
         "Coastal_Cell", "Gap_From_Previous_km"]
    ].round(2).to_string(index=False)
)


Gaps larger than 3 km: 23
Gaps larger than 5 km: 13
Gaps larger than 10 km: 2

Stations with gaps larger than 3 km:
 Previous_No  No Coastal_Cell  Gap_From_Previous_km
           2   3           C1                  6.34
           9  10           C2                  7.02
          15  16           C2                  3.06
          21  22           C3                 25.29
          27  28           C4                  4.25
          28  29           C4                  9.17
          37  38           C5                  8.33
          38  39           C6                  4.20
          66  67           C8                  6.00
          68  69           C8                  3.01
          81  82           C8                  3.61
          82  83           C8                  4.28
          93  94           C8                  4.86
         106 107           C8                  7.33
         107 108           C8                 11.42
         118 119           C9                  3.07


In [4]:
# Pipeline step 4

# Maximum allowed gap between stations
MAX_GAP_KM = 3.0

# Identify where a new coastal segment begins
df_sorted["Start_New_Segment"] = (
    df_sorted["Gap_From_Previous_km"].isna()
    | (df_sorted["Gap_From_Previous_km"] > MAX_GAP_KM)
    | (
        df_sorted["Coastal_Cell"]
        != df_sorted["Coastal_Cell"].shift(1)
    )
)

# Assign segment IDs
df_sorted["Segment_ID"] = (
    df_sorted["Start_New_Segment"].cumsum()
)

# Summarize segments
segment_summary = df_sorted.groupby("Segment_ID").agg(
    Coastal_Cell=("Coastal_Cell", "first"),
    Start_No=("No", "first"),
    End_No=("No", "last"),
    Station_Count=("No", "size")
)

print("Total segments:", len(segment_summary))
print("Total stations retained:", segment_summary["Station_Count"].sum())

print("Single-station segments:",
      (segment_summary["Station_Count"] == 1).sum())

print("\nSegment Summary:")
print(segment_summary.to_string())


Total segments: 27
Total stations retained: 144
Single-station segments: 10

Segment Summary:
           Coastal_Cell  Start_No  End_No  Station_Count
Segment_ID                                              
1                    C1         1       2              2
2                    C1         3       9              7
3                    C2        10      15              6
4                    C2        16      21              6
5                    C3        22      27              6
6                    C4        28      28              1
7                    C4        29      37              9
8                    C5        38      38              1
9                    C6        39      47              9
10                   C7        48      62             15
11                   C8        63      66              4
12                   C8        67      68              2
13                   C8        69      81             13
14                   C8        82      82          

In [5]:
# Pipeline step 5

SPACING_M = 20

# Select consecutive stations within the same segment
# Both stations must be non-protected
eligible_pairs = df_sorted[
    (
        df_sorted["Segment_ID"] ==
        df_sorted["Segment_ID"].shift(1)
    )
    &
    (df_sorted["Is_Protected"] == 0)
    &
    (df_sorted["Is_Protected"].shift(1) == 0)
].copy()

# Estimate how many 20-metre points will be generated
eligible_pairs["Estimated_New_Points"] = (
    np.ceil(
        eligible_pairs["Gap_From_Previous_km"] * 1000
        / SPACING_M
    ).astype(int) - 1
).clip(lower=0)

print("Eligible station pairs:", len(eligible_pairs))

print("Interpolation distance (km):",
      round(eligible_pairs["Gap_From_Previous_km"].sum(), 2))

print("Estimated new points:",
      eligible_pairs["Estimated_New_Points"].sum())

print("Original JPS stations:", len(df_sorted))

print("Estimated total points:",
      len(df_sorted) +
      eligible_pairs["Estimated_New_Points"].sum())


Eligible station pairs: 107
Interpolation distance (km): 53.64
Estimated new points: 2631
Original JPS stations: 144
Estimated total points: 2775


In [6]:
# Pipeline step 6

generated_points = []

for end_idx in eligible_pairs.index:
    start = df_sorted.iloc[end_idx - 1]
    end = df_sorted.iloc[end_idx]

    azimuth, _, distance_m = geod.inv(
        start["Longitude_DD"],
        start["Latitude_DD"],
        end["Longitude_DD"],
        end["Latitude_DD"]
    )

    for d in np.arange(SPACING_M, distance_m, SPACING_M):
        lon, lat, _ = geod.fwd(
            start["Longitude_DD"],
            start["Latitude_DD"],
            azimuth,
            float(d)
        )

        generated_points.append({
            "Start_JPS_No": int(start["No"]),
            "End_JPS_No": int(end["No"]),
            "Segment_ID": int(start["Segment_ID"]),
            "Latitude_DD": lat,
            "Longitude_DD": lon,
            "Distance_From_Start_m": d,
            "Fraction_Between_Stations": d / distance_m
        })

df_generated = pd.DataFrame(generated_points)

print("Generated new points:", len(df_generated))
print("Original JPS stations:", len(df_sorted))
print("Total points:", len(df_generated) + len(df_sorted))

df_generated.head()


Generated new points: 2631
Original JPS stations: 144
Total points: 2775
Out[5]: 
   Start_JPS_No  End_JPS_No  ...  Distance_From_Start_m  Fraction_Between_Stations
0             1           2  ...                   20.0                   0.015542
1             1           2  ...                   40.0                   0.031084
2             1           2  ...                   60.0                   0.046626
3             1           2  ...                   80.0                   0.062168
4             1           2  ...                  100.0                   0.077710

[5 rows x 7 columns]


In [7]:
# Pipeline step 7

# Step 7: Interpolate erosion data for generated points

source = df_sorted.set_index("No")

fraction = df_generated["Fraction_Between_Stations"]

features = [
    "Annual_Erosion_Rate_m_yr",
    "Physical_Erosion_Score",
    "Economic_Score"
]

# Interpolate numerical values
for col in features:

    start_values = df_generated["Start_JPS_No"].map(source[col])
    end_values = df_generated["End_JPS_No"].map(source[col])

    df_generated[col] = (
        start_values +
        (end_values - start_values) * fraction
    ).round(4)

# Calculate estimated total risk score
df_generated["Total_Score_Cleaned"] = (
    df_generated["Physical_Erosion_Score"] *
    df_generated["Economic_Score"]
).round(2)

# Generated points come from non-protected station pairs
df_generated["Is_Protected"] = 0

# Assign coastal cell
df_generated["Coastal_Cell"] = (
    df_generated["Start_JPS_No"].map(source["Coastal_Cell"])
)

# Keep track of nearest original JPS station
df_generated["Nearest_JPS_No"] = np.where(
    fraction <= 0.5,
    df_generated["Start_JPS_No"],
    df_generated["End_JPS_No"]
)

df_generated["District"] = (
    df_generated["Nearest_JPS_No"].map(source["District"])
)

df_generated["Area_Location"] = (
    df_generated["Nearest_JPS_No"].map(source["Area_Location"])
)

df_generated["Source_Type"] = "Interpolated"

print("Generated points:", len(df_generated))

print("\nMissing feature values:")
print(df_generated[features].isnull().sum())

print("\nFirst 5 generated points:")
print(df_generated[
    [
        "Latitude_DD",
        "Longitude_DD",
        "Annual_Erosion_Rate_m_yr",
        "Physical_Erosion_Score",
        "Economic_Score",
        "Total_Score_Cleaned"
    ]
].head())


Generated points: 2631

Missing feature values:
Annual_Erosion_Rate_m_yr    0
Physical_Erosion_Score      0
Economic_Score              0
dtype: int64

First 5 generated points:
   Latitude_DD  Longitude_DD  ...  Economic_Score  Total_Score_Cleaned
0     5.838462    102.546779  ...          3.3969                16.72
1     5.838395    102.546947  ...          3.3938                16.44
2     5.838329    102.547115  ...          3.3907                16.16
3     5.838262    102.547283  ...          3.3876                15.89
4     5.838196    102.547451  ...          3.3845                15.61

[5 rows x 6 columns]


In [8]:
# Pipeline step 8

# Step 8: Combine original and generated coastal points

# Copy original JPS stations
df_original = df_sorted.copy()

# Identify original records
df_original["Source_Type"] = "JPS_Original"

df_original["Start_JPS_No"] = df_original["No"]
df_original["End_JPS_No"] = df_original["No"]
df_original["Nearest_JPS_No"] = df_original["No"]

df_original["Distance_From_Start_m"] = 0.0
df_original["Fraction_Between_Stations"] = 0.0

# Combine both datasets
df_combined = pd.concat(
    [df_original, df_generated],
    ignore_index=True
)

# Sort by coastal segment and station
df_combined = df_combined.sort_values(
    ["Segment_ID", "Start_JPS_No",
     "Fraction_Between_Stations"]
).reset_index(drop=True)

# Assign unique IDs
df_combined["Transect_ID"] = np.arange(
    1, len(df_combined) + 1
)

print("=== COMBINED DATASET ===")
print("Total points:", len(df_combined))

print("\nSource distribution:")
print(df_combined["Source_Type"].value_counts())

print("\nProtected points:",
      df_combined["Is_Protected"].sum())

print("\nMissing erosion rates:",
      df_combined["Annual_Erosion_Rate_m_yr"].isnull().sum())

print("\nFirst 5 rows:")
print(df_combined[
    ["Transect_ID", "Start_JPS_No", "Segment_ID",
     "Source_Type", "Annual_Erosion_Rate_m_yr"]
].head())


=== COMBINED DATASET ===
Total points: 2775

Source distribution:
Source_Type
Interpolated    2631
JPS_Original     144
Name: count, dtype: int64

Protected points: 9

Missing erosion rates: 9

First 5 rows:
   Transect_ID  Start_JPS_No  ...   Source_Type Annual_Erosion_Rate_m_yr
0            1             1  ...  JPS_Original                   4.9400
1            2             1  ...  Interpolated                   4.8632
2            3             1  ...  Interpolated                   4.7864
3            4             1  ...  Interpolated                   4.7097
4            5             1  ...  Interpolated                   4.6329

[5 rows x 5 columns]


In [9]:
# Pipeline step 9

# Step 9: Final dataset verification

features = [
    "Annual_Erosion_Rate_m_yr",
    "Physical_Erosion_Score",
    "Economic_Score"
]

print("=== FINAL TRANSECT VERIFICATION ===")

print("Total points:", len(df_combined))

print("Duplicate Transect IDs:",
      df_combined["Transect_ID"].duplicated().sum())

print("\nMissing coordinates:")
print(df_combined[
    ["Latitude_DD", "Longitude_DD"]
].isnull().sum())

print("\nMissing features in non-protected points:")
print(
    df_combined.loc[
        df_combined["Is_Protected"] == 0,
        features
    ].isnull().sum()
)

print("\nSource distribution:")
print(df_combined["Source_Type"].value_counts())

print("\nProtected points:",
      df_combined["Is_Protected"].sum())

print("\nTotal is correct:",
      len(df_combined) == 2775)

assert df_combined["Transect_ID"].is_unique
assert df_combined[["Latitude_DD", "Longitude_DD"]].notna().all().all()
assert df_combined.loc[df_combined["Is_Protected"].eq(0), features].notna().all().all()
retained_originals = df_combined.loc[df_combined["Source_Type"].eq("JPS_Original"), "No"]
assert len(retained_originals) == 144 and set(retained_originals) == set(df["No"])


=== FINAL TRANSECT VERIFICATION ===
Total points: 2775
Duplicate Transect IDs: 0

Missing coordinates:
Latitude_DD     0
Longitude_DD    0
dtype: int64

Missing features in non-protected points:
Annual_Erosion_Rate_m_yr    0
Physical_Erosion_Score      0
Economic_Score              0
dtype: int64

Source distribution:
Source_Type
Interpolated    2631
JPS_Original     144
Name: count, dtype: int64

Protected points: 9

Total is correct: True


In [10]:
# Pipeline step 10

import os
import pandas as pd

# Create processed folder if needed
os.makedirs(PROJECT_ROOT / "data/processed", exist_ok=True)

# Save new dataset without overwriting the old one
output_path = PROJECT_ROOT / "data/processed/terengganu_coastal_erosion_transects_144.csv"

df_combined.to_csv(output_path, index=False)

# Reload to verify
df_check = pd.read_csv(output_path)

print("Dataset saved successfully!")
print("Total points:", len(df_check))
print("Total columns:", len(df_check.columns))

print("\nSource distribution:")
print(df_check["Source_Type"].value_counts())

print("\nProtected points:")
print(df_check["Is_Protected"].sum())

print("\nFile location:", output_path.relative_to(PROJECT_ROOT))


Dataset saved successfully!
Total points: 2775
Total columns: 29

Source distribution:
Source_Type
Interpolated    2631
JPS_Original     144
Name: count, dtype: int64

Protected points:
9

File location: data/processed/terengganu_coastal_erosion_transects_144.csv
